# A tokenizer from scratch (Byte Pair Encoding)

Start from single letters, count every pair of letters that sit side by side inside a word, glue the most common pair into one new piece, and repeat.

In [ ]:
TEXT = """
The farm sells strawberries and fresh strawberry jam at the market.
We picked a strawberry from the field and ate it in the sun.
The strawberry field is next to the river, behind the old barn.
Rain fell on the farm all night, and the river was high in the morning.
The farmer laid straw under every strawberry plant to keep the fruit dry.
A child at the market asked for one more strawberry.
My mother makes strawberry cake every summer for the whole street.
The barn smells of straw, rain and warm bread.
We sell raspberry jam, blueberry jam and strawberry jam at the same table.
The market opens early, and the strawberry stall is always the first to sell out.
Birds love the strawberry plants, so the farmer puts a net over the field.
In the morning the field is wet, and the straw keeps the berries clean.
A strawberry is red, sweet and a little sour.
The river runs past the farm and under the stone bridge.
We drank tea by the river and shared a box of strawberries.
The farmer said this was the best strawberry summer in years.
After the rain the berries were sweeter than ever.
She wrote the price of every berry on a small board at the market.
The strawberry plants need sun, water and a lot of patience.
We walked home along the river with red hands and a full basket.
The baker in the street makes strawberry tarts on Saturday.
There is straw on the floor of the barn and on the path to the field.
Every strawberry we picked went into the same old basket.
The farm dog sleeps in the barn on a bed of straw.
At the end of the summer the field is quiet and the market closes.
The children counted every strawberry in the basket.
Berries from the farm are sold at the market every week.
The weather was warm, the river was calm, and the strawberry stall was busy.
"""

## Letters, words

In [ ]:
import re
from collections import Counter

words = re.findall(r"[a-z]+", TEXT.lower())
letters = [c for w in words for c in w]
print(len(TEXT.strip().splitlines()), "sentences,", len(words), "words,", len(letters), "letters")
print(len(set(words)), "different words")
print("strawberry appears", words.count("strawberry"), "times")

28 sentences, 335 words, 1398 letters
142 different words
strawberry appears 16 times


## A whole-word list can't read a word it never saw

In [ ]:
vocab_words = set(words)
test = "the strawbery stall sells blackberry jam"
print(" ".join(w if w in vocab_words else "[?]" for w in test.split()))

the [?] stall sells [?] jam


## Glue the most common pair, again and again

In [ ]:
freq = Counter(words)
split = {w: list(w) for w in freq}
merges = []
for _ in range(150):
    pairs = Counter()
    for w, f in freq.items():
        s = split[w]
        for a, b in zip(s, s[1:]):
            pairs[(a, b)] += f
    if not pairs:
        break
    (a, b), n = pairs.most_common(1)[0]
    merges.append((a, b, n))
    for w in split:
        s, out, i = split[w], [], 0
        while i < len(s):
            if i + 1 < len(s) and s[i] == a and s[i + 1] == b:
                out.append(a + b); i += 2
            else:
                out.append(s[i]); i += 1
        split[w] = out

for a, b, n in merges[:4]:
    print(f"{a} + {b} -> {a + b}   ({n} times)")

t + h -> th   (65 times)
th + e -> the   (61 times)
e + r -> er   (51 times)
s + t -> st   (31 times)


## Watch one word get glued

In [ ]:
def tokenize(word, upto=None):
    parts = list(word)
    for a, b, _ in merges[:upto]:
        out, i = [], 0
        while i < len(parts):
            if i + 1 < len(parts) and parts[i] == a and parts[i + 1] == b:
                out.append(a + b); i += 2
            else:
                out.append(parts[i]); i += 1
        parts = out
    return parts

last = None
for n in range(len(merges) + 1):
    now = tokenize("strawberry", n)
    if now != last:
        print(f"after {n:3d} glues:", " | ".join(now))
        last = now

after   0 glues: s | t | r | a | w | b | e | r | r | y
after   3 glues: s | t | r | a | w | b | er | r | y
after   4 glues: st | r | a | w | b | er | r | y
after   5 glues: st | ra | w | b | er | r | y
after   7 glues: st | ra | w | ber | r | y
after   8 glues: st | ra | w | berr | y
after   9 glues: stra | w | berr | y
after  10 glues: straw | berr | y
after  12 glues: straw | berry
after  15 glues: strawberry


## Every word fits now

In [ ]:
for w in ["the", "strawberry", "strawbery", "blackberry", "raspberry"]:
    print(f"{w:12}", " | ".join(tokenize(w)))

print(len(letters), "letters ->", sum(len(split[w]) * f for w, f in freq.items()), "tokens")

the          the
strawberry   strawberry
strawbery    straw | ber | y
blackberry   b | l | a | c | k | berry
raspberry    ra | s | p | berry
1398 letters -> 485 tokens


## Try your own

Change TEXT above (or add a word to it many times), run all the cells again, and tokenize your own word here.

In [ ]:
print(" | ".join(tokenize("blueberry")))

b | l | u | e | berry
